In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Ecommerce Data Platform")
    .master("spark://spark-master:7077")
    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1",
            "io.delta:delta-spark_2.12:3.2.0"
        ])
    )
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .getOrCreate()
)


spark.sparkContext.setLogLevel("ERROR")



: 

In [5]:
spark

: 

In [16]:
spark.sql("CREATE DATABASE IF NOT EXISTS test_db")


DataFrame[]

In [15]:
spark.sql("SHOW TABLES IN test_db").show()

AnalysisException: [SCHEMA_NOT_FOUND] The schema `test_db` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a catalog, verify the current_schema() output, or qualify the name with the correct catalog.
To tolerate the error on drop use DROP SCHEMA IF EXISTS.

In [9]:
spark.sql("SHOW DATABASES").show()

In [21]:
spark.sql("DROP TABLE IF EXISTS test_db.sample_orders")

In [23]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS test_db.sample_orders2(
        name STRING,
        age INT,
        salary INT
    )
    USING DELTA
    LOCATION '/opt/spark-data/test/sample_orders'
""")

In [17]:
df = spark.read.csv("/opt/spark-data/input/data.csv", header=True, inferSchema=True)
df.show()

+-------+---+------+
|   name|age|salary|
+-------+---+------+
| manish| 28| 55000|
| ramesh| 32| 48000|
|  rohan| 26| 61000|
|  sapna| 30| 72000|
|shivani| 27| 68000|
+-------+---+------+



In [18]:
df.write.format("delta").mode("append").save("/opt/spark-data/test/sample_orders")

In [21]:
spark.sql("select * from test_db.sample_orders").show()

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `test_db`.`sample_orders` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 14;
'Project [*]
+- 'UnresolvedRelation [test_db, sample_orders], [], false


In [31]:
spark.sql("DELETE FROM test_db.sample_orders2 WHERE name = 'ramesh'")

In [32]:
spark.sql("DESCRIBE HISTORY test_db.sample_orders2").show()